# Capstone — Search Intelligence: prioritizing content for human review

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring  
**Research question:** Can historical search-performance signals help prioritize content items for human review, and does Logistic Regression improve on a transparent rule baseline?  
**Decision supported:** Prioritize investigation—not automatically refresh a page.

This notebook consolidates the evidence from Weeks 3–7 into a repeatable workflow. The main warehouse experiment predicts an April 2026 impressions decline from March-only features. A separate 30,000-item starter-data playbook ranks current-window CTR opportunities. These are different analyses and their populations and metrics must not be mixed.

**Privacy:** Aggregate metrics only. Do not print client IDs, content IDs, URLs or private queries. Run this only in an environment permitted by FlyRank's data-use terms. Never commit tokens or raw exports.

## 1. Question

Content and SEO teams have limited time to investigate pages when search visibility changes. Reviewing every item equally is inefficient, but low CTR or declining impressions do not prove that content is bad or needs a refresh.

**Question:** Using information available by the end of March 2026, can five search-performance features rank content items whose April impressions decline by more than 20%, and does Logistic Regression improve on the Week-4 transparent rule?

**Operational output:** A ranked investigation queue. A human should check intent, SERP context, rankings, demand and measurement coverage before deciding what to change. This is a prediction/ranking experiment, not a causal study of content refreshes.

In [ ]:
FEATURE_WINDOW = "2026-03-01 through 2026-03-31"
OUTCOME_WINDOW = "2026-04-01 through 2026-04-30"
FEATURES = [
    "march_impressions", "march_clicks", "march_mean_position",
    "march_impression_days", "march_active_days_span",
]
FORBIDDEN_FEATURES = {
    "april_impressions", "april_trend_pct", "april_decline_label",
    "client_hash_id", "content_hash_id",
}
assert not (set(FEATURES) & FORBIDDEN_FEATURES)
print("Feature window:", FEATURE_WINDOW)
print("Outcome window:", OUTCOME_WINDOW)
print("Target: April impressions decline greater than 20% relative to March.")
print("Feature count:", len(FEATURES), "| forbidden predictors excluded.")

## 2. Data

**Release:** Gated FlyRank internship warehouse release v20260703, accessed through Hugging Face.  
**Main table:** fact_content_daily_performance; content/client dimensions are contextual, and pseudonymous client identifiers are used for grouping only.  
**Feature window:** March 2026. **Outcome window:** April 2026.  
**Main feature frame:** 176,737 content items across 47 pseudonymized clients; overall April-decline prevalence is 53.2%.  
**Exclusions:** April-derived predictors, target/trend fields and identifiers as model features. The analysis focuses on Google Search Console metrics; GA4 zeros are not interpreted as zero engagement when availability is not confirmed.

A separate starter dataset contains 30,000 anonymized items across 32 pseudonymized clients with trailing-90-day metrics. It supports the rule-based action playbook only, not the warehouse model metrics.

In Colab, add a Hugging Face read token as a secret named HF_TOKEN. The token is read from Colab Secrets or the HF_TOKEN environment variable and must never be written into a code cell.

In [ ]:
# Build the March feature frame and April label from the gated warehouse.
%pip -q install duckdb huggingface_hub scikit-learn pandas matplotlib
import os, duckdb, numpy as np, pandas as pd
from IPython.display import display

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN missing. Add it through Colab Secrets; do not paste it into the notebook.")

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

feature_sql = f"""
WITH march AS (
 SELECT client_hash_id, content_hash_id,
  SUM(gsc_impressions) AS march_impressions,
  SUM(gsc_clicks) AS march_clicks,
  AVG(NULLIF(gsc_avg_position, 0)) AS march_mean_position,
  COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS march_impression_days,
  DATE_DIFF('day', MIN(report_date), MAX(report_date)) + 1 AS march_active_days_span
 FROM {FACT_MARCH} GROUP BY 1,2
),
april AS (
 SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS april_impressions
 FROM {FACT_APRIL} GROUP BY 1,2
)
SELECT m.*, a.april_impressions,
  100.0 * (a.april_impressions - m.march_impressions) / NULLIF(m.march_impressions, 0) AS april_trend_pct
FROM march m JOIN april a USING (client_hash_id, content_hash_id)
WHERE m.march_impressions > 0
"""
feature_frame = con.sql(feature_sql).df()
feature_frame["april_decline_label"] = (feature_frame["april_trend_pct"] < -20.0).astype(int)
X = feature_frame[FEATURES].copy()
y = feature_frame["april_decline_label"].astype(int)
groups = feature_frame["client_hash_id"].copy()

print(f"Feature items: {len(feature_frame):,}")
print(f"Pseudonymized clients: {groups.nunique():,}")
print(f"Overall decline prevalence: {y.mean():.1%}")
print(f"Missing feature values: {int(X.isna().sum().sum()):,}")
assert len(feature_frame) == 176737, "Unexpected row count; investigate before using published metrics."
assert groups.nunique() == 47, "Unexpected client count; investigate before using published metrics."
assert feature_frame["content_hash_id"].is_unique
assert X.columns.tolist() == FEATURES
assert y.isin([0, 1]).all()
print("Data-contract checks passed.")

## 3. Methodology

**Features and label:** Five March-only variables—impressions, clicks, mean position (excluding zero position), days with impressions, and active-day span. The positive label means April impressions declined by more than 20% relative to March.

**Baseline:** The Week-4 rule rebuilt from March-only features. It prioritizes at least 500 impressions, average position greater than 0 and at most 20, and CTR below 0.5%. CTR is calculated as clicks / impressions × 100.

**Model:** Logistic Regression with median imputation and standardization fitted inside a scikit-learn pipeline on training data only.

**Validation:** Client-grouped 75/25 holdout using GroupShuffleSplit with random_state=42. Both methods are scored on the same held-out rows. This tests held-out-client generalization, not a later-calendar-period test.

**Leakage checks:** April fields, target/trend fields and identifiers are excluded from model features. The Week-3 deliberate leakage test showed a perfect AUC when an April-derived field was included; that field is excluded and the perfect score is treated as a warning.

In [ ]:
# Client-grouped split, model training, baseline and metrics.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

model_df = feature_frame.reset_index(drop=True).copy()
X = model_df[FEATURES].copy()
y = model_df["april_decline_label"].astype(int)
groups = model_df["client_hash_id"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]
client_overlap = set(groups_train.unique()) & set(groups_test.unique())
assert len(client_overlap) == 0
assert set(train_idx).isdisjoint(set(test_idx))

model = Pipeline([
 ("imputer", SimpleImputer(strategy="median")),
 ("scaler", StandardScaler()),
 ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
model.fit(X_train, y_train)
model_prob = model.predict_proba(X_test)[:, 1]

march_ctr = (100 * X_test["march_clicks"] / X_test["march_impressions"].replace(0, np.nan)).fillna(0)
ctr_gap = np.clip(0.5 - march_ctr, 0, 0.5) / 0.5
visibility_gate = ((X_test["march_impressions"] >= 500) & (X_test["march_mean_position"] > 0) & (X_test["march_mean_position"] <= 20)).astype(int)
baseline_score = (np.log1p(X_test["march_impressions"]) * ctr_gap * visibility_gate).to_numpy()

def precision_at_k(y_true, scores, k):
    y_arr = np.asarray(y_true)
    idx = np.argsort(-np.asarray(scores), kind="mergesort")[:min(k, len(y_arr))]
    return float(y_arr[idx].mean())

comparison = pd.DataFrame([
 {"method":"Rule baseline", "Precision@20":precision_at_k(y_test, baseline_score, 20),
  "Precision@50":precision_at_k(y_test, baseline_score, 50),
  "ROC-AUC":roc_auc_score(y_test, baseline_score),
  "Average precision":average_precision_score(y_test, baseline_score)},
 {"method":"Logistic Regression", "Precision@20":precision_at_k(y_test, model_prob, 20),
  "Precision@50":precision_at_k(y_test, model_prob, 50),
  "ROC-AUC":roc_auc_score(y_test, model_prob),
  "Average precision":average_precision_score(y_test, model_prob)}
])
print(f"Train rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Train clients: {groups_train.nunique()}; held-out clients: {groups_test.nunique()}")
print(f"Client overlap: {len(client_overlap)}")
print(f"Overall prevalence: {y.mean():.1%}; test prevalence: {y_test.mean():.1%}")
display(comparison.round(4))
assert len(X_train) == 133473 and len(X_test) == 43264
assert groups_train.nunique() == 35 and groups_test.nunique() == 12
print("Grouped split checks passed.")

## 4. Results — model versus baseline

The client-grouped test set contains 43,264 items across 12 held-out clients. Its positive-label prevalence is 45.3%; the full feature-frame prevalence is 53.2%. These are different quantities and should not be interchanged.

- Precision@20 ties: 0.650 versus 0.650.
- Precision@50 favors the rule baseline: 0.560 versus 0.540.
- Logistic Regression has higher ROC-AUC (0.578 versus 0.518) and average precision (0.515 versus 0.475).
- Evidence is mixed. For a small review queue, the model has not demonstrated an operational advantage over the transparent rule.

Charts generated below describe ranking metrics; they do not show causal impact or guaranteed business value.

In [ ]:
import matplotlib.pyplot as plt
metrics = ["Precision@20", "Precision@50", "ROC-AUC", "Average precision"]
b = comparison.loc[comparison["method"] == "Rule baseline"].iloc[0]
m = comparison.loc[comparison["method"] == "Logistic Regression"].iloc[0]
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(metrics)); width = 0.36
ax.bar(x - width/2, [b[k] for k in metrics], width, label="Rule baseline")
ax.bar(x + width/2, [m[k] for k in metrics], width, label="Logistic Regression")
ax.set_xticks(x, metrics, rotation=12, ha="right")
ax.set_ylim(0, 0.8); ax.set_ylabel("Score")
ax.set_title("Client-grouped holdout: model versus baseline")
ax.legend(); fig.tight_layout(); plt.show()

aggregate_result = {
 "feature_frame_rows": int(len(feature_frame)), "client_count": int(groups.nunique()),
 "train_rows": int(len(X_train)), "test_rows": int(len(X_test)),
 "train_clients": int(groups_train.nunique()), "test_clients": int(groups_test.nunique()),
 "overall_decline_prevalence": float(y.mean()), "test_decline_prevalence": float(y_test.mean()),
 "client_overlap": int(len(client_overlap)), "random_state": 42,
 "metrics": comparison.round(4).to_dict(orient="records")
}
display(pd.DataFrame(aggregate_result["metrics"]))
print("Aggregate-only metrics; no item-level predictions are exported.")

## 5. Limitations and honest framing

1. The label is a threshold-based month-over-month change and may be affected by seasonality, demand shifts, ranking changes, indexing and measurement coverage.
2. The panel is unbalanced; the eligible population may not represent every page or client.
3. Client-grouped validation tests unseen-client generalization for this March-to-April task, not future-period performance. June 2026 is identified as a sealed period and these results are not June test results.
4. Five aggregate features cannot explain every decline or prediction error. Low CTR is not proof of weak content.
5. The comparison focuses on one learned model and one transparent baseline. Repeated temporal validation and more model comparisons are needed before production use.
6. No causal claim is supported: this analysis does not show that refreshing content causes impressions or clicks to recover.

**Claim boundary:** observed ranking performance is decision-support evidence, not proof of content quality, a diagnosis of Google's algorithm, or a guarantee of business uplift.

In [ ]:
assert set(FEATURES).isdisjoint(FORBIDDEN_FEATURES)
assert len(client_overlap) == 0
assert len(X_train) + len(X_test) == len(feature_frame)
assert comparison["method"].tolist() == ["Rule baseline", "Logistic Regression"]
assert (comparison[["Precision@20", "Precision@50", "ROC-AUC", "Average precision"]] >= 0).all().all()
assert (comparison[["Precision@20", "Precision@50", "ROC-AUC", "Average precision"]] <= 1).all().all()
print("Integrity checks passed.")
print("A grouped client split is not a future-period holdout.")

## 6. Ranked recommendations

These recommendations use the separate starter-data action playbook. They are not derived from the main model's April-decline probabilities.

1. **Keep the transparent rule as the primary review queue.** It selected 9,759 of 30,000 starter-data items (32.53%), using at least 500 impressions, average position in (0, 20], and CTR below 0.5%.
2. **Require human review before changing content.** Inspect query intent, snippet relevance, SERP features, accuracy, completeness, ranking movement, demand and measurement coverage.
3. **Do not use update recency alone.** The signal audit found only a weak relationship between update recency and CTR; age should be context, not a trigger.
4. **Evaluate later outcomes before claiming impact.** Use aligned, predeclared windows and a held-out temporal test; before/after movement alone is not causal evidence.
5. **Monitor the queue.** Track missingness, CTR rate units, position-zero handling, selection share, score distribution and reason-code frequency.

The next cell reads only the aggregate action-playbook receipt; it does not load or display the item-level queue.

In [ ]:
import json
from urllib.request import urlopen
metrics_url = "https://raw.githubusercontent.com/Asif-Ahmed-Rezvi/flyrank-internship-ml/main/work/outputs/action_playbook_metrics.json"
with urlopen(metrics_url, timeout=30) as response:
    action_metrics = json.loads(response.read().decode("utf-8"))
assert action_metrics["input_rows"] == 30000
assert action_metrics["pages_selected"] == 9759
assert action_metrics["pages_not_selected"] == 20241
assert action_metrics["selection_rule"]["minimum_impressions_90d"] == 500
assert action_metrics["selection_rule"]["average_position_max_inclusive"] == 20
assert action_metrics["selection_rule"]["ctr_threshold_pct"] == 0.5
summary = pd.DataFrame([
 {"Measure":"Starter rows ranked", "Value":action_metrics["pages_ranked"]},
 {"Measure":"Selected for human review", "Value":action_metrics["pages_selected"]},
 {"Measure":"Not selected", "Value":action_metrics["pages_not_selected"]},
 {"Measure":"Selected share (%)", "Value":action_metrics["selected_share_pct"]},
 {"Measure":"Median selected score", "Value":action_metrics["selected_score_summary"]["median"]},
])
display(summary)
print("Reason-code counts:", action_metrics["reason_code_counts"])
print("Aggregate action-playbook receipt checks passed.")

## 7. Paper artifacts

The deployed paper includes two inline SVG charts:
- Top-ranked precision: Precision@20 and Precision@50, comparing baseline and Logistic Regression.
- Ranking discrimination: ROC-AUC and average precision.

The precision chart shows a top-20 tie and a slight baseline advantage at top 50. The ranking-discrimination chart shows higher ROC-AUC and average precision for Logistic Regression. The paper separately reports 9,759 selected and 20,241 not selected in the starter-data playbook; those counts are not from the warehouse model test set.

No client-level or content-level examples are included in the public paper.

In [ ]:
PAPER_URL = "https://asif-ahmed-rezvi.github.io/flyrank-internship-ml/"
REPO_URL = "https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml"
DATA_CREDIT_URL = "https://flyrank.ai"
assert PAPER_URL.startswith("https://")
assert REPO_URL.startswith("https://github.com/")
assert DATA_CREDIT_URL == "https://flyrank.ai"
print("Paper:", PAPER_URL)
print("Repository:", REPO_URL)
print("Data credit:", DATA_CREDIT_URL)
print("Required URL receipt: submission/paper_url.txt (exactly one URL line).")

## 8. Reproducibility

- Repository: [flyrank-internship-ml](https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml)
- Deployed paper: [Read the research paper](https://asif-ahmed-rezvi.github.io/flyrank-internship-ml/)
- Data contract: [Week 3](https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb)
- Leakage checks: [Week 3](https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)
- Rule baseline: [Week 4](https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb)
- Model experiment: [Week 5](https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb)
- Validation audit: [Week 6](https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb)
- Action playbook: [Week 7](https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb)
- Capstone: [this notebook](https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb)
- URL receipt: [submission/paper_url.txt](https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/submission/paper_url.txt)

To reproduce, use authorized Hugging Face access, provide HF_TOKEN through Colab Secrets or an environment variable, and run all cells top to bottom. Split seed: 42. The notebook does not commit credentials, raw exports, page-level predictions or private queries.

## 9. Acknowledgments and data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai). Thanks to FlyRank for providing the dataset and learning context for this applied research exercise.

This is an internship research artifact for learning and decision support. It is not an official FlyRank product claim or a guarantee of outcomes for any individual page.

## Self-check

- [x] Research question and supported decision are explicit.
- [x] Data release, tables, windows and exclusions are documented in public-safe language.
- [x] A repeatable warehouse query, grouped split, baseline, model and metric calculations are included.
- [x] Leakage guardrails and client-overlap assertions are included.
- [x] Ranked recommendations are connected to the separate aggregate action-playbook receipt.
- [x] The paper includes all nine required sections and the FlyRank data-credit link.
- [x] Five-minute demo outline and two shareable cuts are included below.
- [ ] **Run all cells in authorized Colab and verify the current warehouse outputs before final submission.** This gated-data notebook has not been executed in your Colab session.
- [ ] Confirm GitHub Pages finishes publishing the latest paper-source commit.

## 5-minute demo outline (Week-8 showcase)

**0:00–0:40 — The decision.** Explain that content teams need to prioritize investigation when search performance shifts; the model does not automatically decide which pages to refresh.

**0:40–1:30 — Data and method.** Show the March-only feature window, April decline label, five contracted features, transparent baseline and client-grouped 75/25 holdout. April fields are used only to define the outcome.

**1:30–2:30 — Results.** Show the paper's two charts. Precision@20 ties at 0.650; Precision@50 favors the baseline (0.560 vs 0.540); ROC-AUC and average precision favor Logistic Regression. Call this mixed evidence, not a model victory.

**2:30–3:30 — Action playbook.** In the separate 30,000-item starter dataset, 9,759 pages met the transparent review rule. Explain the impression threshold, top-20 average position, CTR below 0.5%, and the high-visibility/low-CTR reason code.

**3:30–4:30 — Recommendations.** Start with the transparent queue; investigate intent, snippet relevance, SERP features, ranking changes, demand and measurement before changing content. Do not use age alone as a refresh trigger.

**4:30–5:00 — Limits and close.** The analysis is observational, the client-grouped split is not a future-period test, and no result proves a refresh will recover impressions. Close with the paper and repository links.

## Shareable cuts

### Short social post

For my FlyRank ML internship capstone, I tested whether March search-performance signals could help rank content items whose impressions declined in April. On a client-grouped holdout, Logistic Regression improved ROC-AUC and average precision, but tied the transparent baseline at Precision@20 and scored slightly lower at Precision@50. The takeaway: use search signals to prioritize investigation, keep the rule-based queue for now, and let human review decide what to change.

Research paper: https://asif-ahmed-rezvi.github.io/flyrank-internship-ml/

### Employer-facing summary (3 sentences)

I built a repeatable search-performance analysis that compares a transparent rule-based review queue with Logistic Regression using five March-only features and a client-grouped holdout. The warehouse experiment covered 176,737 content items across 47 pseudonymized clients, while a separate 30,000-item starter-data analysis produced a reason-coded queue that selected 9,759 items for human review. The model tied the baseline at Precision@20, scored slightly lower at Precision@50, and achieved higher ROC-AUC and average precision, supporting cautious decision-support use rather than automatic refresh decisions.